In [1]:
import os
import pandas as pd
import numpy as np

# Define folder paths relative to this notebook
raw_dir = 'data/raw'
processed_dir = 'data/processed'

# Create folders if they don't exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Define the sample data
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV in raw data folder
csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')

Sample dataset created and saved to data/raw/sample_data.csv


## Load Raw Dataset

In [2]:
df = pd.read_csv('data/raw/sample_data.csv')
df.head()

,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         6 non-null      float64
 1   income      4 non-null      float64
 2   score       6 non-null      float64
 3   zipcode     7 non-null      int64  
 4   city        7 non-null      str    
 5   extra_data  2 non-null      float64
dtypes: float64(4), int64(1), str(1)
memory usage: 518.0 bytes


In [5]:
df.isna().sum()

age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64

## Apply Cleaning Functions

In [3]:
from src.cleaning import fill_missing_median, drop_missing, normalize_data

In [6]:
df_clean = fill_missing_median(df)
df_clean = drop_missing(df_clean)
df_clean = normalize_data(df_clean, method="minmax")
df_clean.head()

,age,income,score,zipcode,city,extra_data
0,0.238095,0.8125,0.653846,0.953688,Beverly,0.5
1,0.761905,0.6250,1.000000,0.000000,New York,1.0
2,0.000000,0.0000,0.596154,0.601791,Chicago,0.5
3,1.000000,1.0000,0.423077,0.999976,SF,0.5
4,0.428571,0.6250,0.884615,0.752640,Austin,0.5


# Save cleaned dataset

In [7]:
from pathlib import Path

output_path = Path("data/processed") / "cleaned_data.csv"
df_clean.to_csv(output_path, index=False)

# Compare original vs cleaned

In [8]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (7, 6)
Cleaned shape: (7, 6)


In [9]:
print("Original missing values:")
print(df.isna().sum())

print("\nCleaned missing values:")
print(df_clean.isna().sum())

Original missing values:
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64

Cleaned missing values:
age           0
income        0
score         0
zipcode       0
city          0
extra_data    0
dtype: int64


In [10]:
print("Original:")
display(df.describe())

print("Cleaned:")
display(df_clean.describe())

Original:


,age,income,score,zipcode,extra_data
count,6.000000,4.000000,6.000000,7.00000,2.000000
mean,39.500000,51000.000000,0.801667,62097.00000,23.500000
std,7.556454,7071.067812,0.092826,36869.63632,26.162951
min,29.000000,42000.000000,0.650000,10001.00000,5.000000
25%,35.000000,47250.000000,0.767500,36479.50000,14.250000
50%,39.500000,52000.000000,0.805000,73301.00000,23.500000
75%,44.000000,55750.000000,0.865000,92156.50000,32.750000
max,50.000000,58000.000000,0.910000,94105.00000,42.000000


Cleaned:


,age,income,score,zipcode,extra_data
count,7.000000,7.000000,7.000000,7.000000,7.000000
mean,0.500000,0.589286,0.585165,0.619424,0.500000
std,0.328479,0.314281,0.325952,0.438381,0.288675
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.333333,0.531250,0.480769,0.314830,0.500000
50%,0.500000,0.625000,0.596154,0.752640,0.500000
75%,0.666667,0.718750,0.769231,0.976832,0.500000
max,1.000000,1.000000,1.000000,1.000000,1.000000


# Cleaning Assumptions

- Missing numeric values are filled with the median because the median is less sensitive to outliers than the mean.
- Rows with remaining missing values are dropped because they are assumed not to be critical to the analysis.
- Numeric columns are normalized using MinMax scaling so that values are placed on a comparable scale.
- The cleaning functions assume that future datasets have a similar column structure and data types.